# SAAM Project — Part I, Section 1: Data Cleaning
**Group AT** — North America (AMER) / Scope 1+2

This notebook performs all data cleaning steps on the raw Datastream output:
1. Load raw data files
2. Remove Datastream error rows and missing ISINs
3. Filter to assigned region (AMER)
4. Apply low-price filter (RI < 0.5 → NaN)
5. Compute monthly simple returns
6. Detect delisted firms and assign return = −100%
7. Forward-fill missing carbon and revenue data
8. Build yearly investment sets (price, stale, carbon criteria)
9. Prepare risk-free rate
10. Save all cleaned data to CSV files

## 0. Setup and Configuration

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ──────────────────────────────────────────────────────────
REGION              = 'AMER'    # North America
SCOPE               = '1+2'    # Scope 1 + Scope 2
LOW_PRICE_THRESHOLD = 0.5      # RI below this → treated as missing
STALE_THRESHOLD     = 0.50     # >50% zero returns over 10y → exclude
MIN_RETURN_MONTHS   = 36       # at least 3 years of returns required
ESTIMATION_WINDOW   = 120      # 10 years × 12 months
Y0                  = 2013     # first allocation year
Y_END               = 2024     # last allocation year
DATA_DIR            = './'     # path to data files

## 1. Load Raw Data

In [ ]:
static     = pd.read_excel(f'{DATA_DIR}Static_2025.xlsx')
ri_m_raw   = pd.read_excel(f'{DATA_DIR}DS_RI_T_USD_M_2025.xlsx')
mv_m_raw   = pd.read_excel(f'{DATA_DIR}DS_MV_T_USD_M_2025.xlsx')
mv_y_raw   = pd.read_excel(f'{DATA_DIR}DS_MV_T_USD_Y_2025.xlsx')
co2_s1_raw = pd.read_excel(f'{DATA_DIR}DS_CO2_SCOPE_1_Y_2025.xlsx')
co2_s2_raw = pd.read_excel(f'{DATA_DIR}DS_CO2_SCOPE_2_Y_2025.xlsx')
rev_raw    = pd.read_excel(f'{DATA_DIR}DS_REV_Y_2025.xlsx')
rf_raw     = pd.read_excel(f'{DATA_DIR}Risk_Free_Rate_2025.xlsx')

print(f"Static:        {static.shape[0]} firms")
print(f"RI monthly:    {ri_m_raw.shape}")
print(f"MV monthly:    {mv_m_raw.shape}")
print(f"MV yearly:     {mv_y_raw.shape}")
print(f"CO2 Scope 1:   {co2_s1_raw.shape}")
print(f"CO2 Scope 2:   {co2_s2_raw.shape}")
print(f"Revenue:       {rev_raw.shape}")
print(f"Risk-free:     {rf_raw.shape[0]} months")

## 2. Remove Datastream Error Rows and Missing ISINs

Datastream inserts error rows (containing `$$ER: E100,...`) when an ISIN cannot be matched. Some rows also have `NaN` as ISIN. We remove both from all datasets.

In [ ]:
def clean_monthly(df_raw, label):
    """Clean monthly Datastream file: remove errors, set ISIN as index."""
    df = df_raw[df_raw['ISIN'].notna()].copy()
    # Remove rows with Datastream error strings in any column
    error_mask = df.apply(
        lambda col: col.astype(str).str.contains(r'\$\$ER', na=False)
        if col.dtype == object else pd.Series(False, index=col.index),
        axis=0
    ).any(axis=1)
    df = df[~error_mask]
    print(f"  {label}: {len(df_raw)} → {len(df)} rows (removed {len(df_raw) - len(df)})")
    names = df.set_index('ISIN')['NAME']  # save firm names
    df = df.set_index('ISIN').drop(columns=['NAME'])
    df = df.apply(pd.to_numeric, errors='coerce')
    return df, names


def clean_annual(df_raw, label):
    """Clean annual Datastream file: remove errors, set ISIN as index."""
    df = df_raw[df_raw['ISIN'].notna()].copy()
    error_mask = df.apply(
        lambda col: col.astype(str).str.contains(r'\$\$ER', na=False)
        if col.dtype == object else pd.Series(False, index=col.index),
        axis=0
    ).any(axis=1)
    df = df[~error_mask]
    print(f"  {label}: {len(df_raw)} → {len(df)} rows (removed {len(df_raw) - len(df)})")
    df = df.set_index('ISIN').drop(columns=['NAME'], errors='ignore')
    df.columns = [int(c) for c in df.columns]
    df = df.apply(pd.to_numeric, errors='coerce')
    return df


ri_m, firm_names = clean_monthly(ri_m_raw, 'RI monthly')
mv_m, _          = clean_monthly(mv_m_raw, 'MV monthly')
co2_s1           = clean_annual(co2_s1_raw, 'CO2 Scope 1')
co2_s2           = clean_annual(co2_s2_raw, 'CO2 Scope 2')
rev              = clean_annual(rev_raw,    'Revenue')
mv_y             = clean_annual(mv_y_raw,   'MV yearly')

dates_m = ri_m.columns.tolist()
print(f"\n  Monthly dates: {dates_m[0].strftime('%Y-%m')} → {dates_m[-1].strftime('%Y-%m')} ({len(dates_m)} months)")

## 3. Filter to Assigned Region (AMER)

We keep only firms that belong to the AMER region **and** appear in all datasets (RI, MV, CO2, Revenue). Firms missing from any file are dropped.

In [ ]:
region_isins = set(static.loc[static['Region'] == REGION, 'ISIN'])
print(f"Firms in Static with Region={REGION}: {len(region_isins)}")

# Intersection: firms present in ALL datasets
common_isins = sorted(
    region_isins
    & set(ri_m.index) & set(mv_m.index)
    & set(co2_s1.index) & set(co2_s2.index)
    & set(rev.index) & set(mv_y.index)
)
print(f"Firms in AMER present in ALL files: {len(common_isins)}")

# Diagnostic: who is missing?
missing_from_ri = region_isins - set(ri_m.index)
if missing_from_ri:
    print(f"  ⚠ {len(missing_from_ri)} AMER firms in Static but missing from RI monthly")

# Apply filter to all datasets
ri_m   = ri_m.loc[common_isins]
mv_m   = mv_m.loc[common_isins]
co2_s1 = co2_s1.loc[common_isins]
co2_s2 = co2_s2.loc[common_isins]
rev    = rev.loc[common_isins]
mv_y   = mv_y.loc[common_isins]
firm_names = firm_names.loc[firm_names.index.isin(common_isins)]

print(f"All datasets filtered to {len(common_isins)} AMER firms")

## 4. Low-Price Filter (RI < 0.5 → NaN)

As suggested in the project instructions, total return indices below 0.5 are treated as missing values. This prevents computing extreme or infinite returns from near-zero prices (e.g., Datastream rounding values below 0.05 to 0).

In [ ]:
low_price_mask = (ri_m < LOW_PRICE_THRESHOLD) & ri_m.notna()
n_low = low_price_mask.sum().sum()
n_zero = (ri_m == 0).sum().sum()
print(f"Prices below {LOW_PRICE_THRESHOLD}: {n_low} cells (of which {n_zero} exactly 0)")

ri_m = ri_m.where(~low_price_mask)  # set low prices to NaN
print(f"Total NaN in RI after filter: {ri_m.isna().sum().sum()}")

## 5. Compute Monthly Simple Returns

$R_{i,t} = \frac{P_{i,t}}{P_{i,t-1}} - 1$

The first column (Dec 1999) is the base price — no return can be computed for it, so we drop it.

In [ ]:
returns_m = ri_m.pct_change(axis=1).iloc[:, 1:]  # drop first col (no return)
dates_ret = returns_m.columns.tolist()

print(f"Returns matrix: {returns_m.shape[0]} firms × {returns_m.shape[1]} months")
print(f"Date range: {dates_ret[0].strftime('%Y-%m')} → {dates_ret[-1].strftime('%Y-%m')}")

# Sanity check: extreme returns
n_extreme = (returns_m.abs() > 5).sum().sum()
print(f"Extreme returns (|R| > 500%): {n_extreme}")
if n_extreme > 0:
    # Show them for inspection
    mask = returns_m.abs() > 5
    for isin in mask.index[mask.any(axis=1)]:
        cols = mask.columns[mask.loc[isin]]
        for c in cols:
            print(f"  {isin} ({firm_names.get(isin, '?')}): {c.strftime('%Y-%m')} → R = {returns_m.loc[isin, c]:.2%}")

## 6. Detect Delisted Firms (Return = −100%)

When a firm is delisted, Datastream stops reporting prices (RI becomes permanently NaN). The project instructs us to acknowledge a realized return of −100% at the delisting date.

**Detection rule:** for each firm, find the first month where the price transitions from a valid value to NaN and *all subsequent months remain NaN* (permanent disappearance). At that transition, set the return to −1.

In [ ]:
# Vectorized: find, for each firm, the index of the last valid (non-NaN) price
prices_arr = ri_m.values  # (N_firms, N_dates)
valid_mask = ~np.isnan(prices_arr)  # True where price is valid

delisted_firms = []

for i, isin in enumerate(ri_m.index):
    row = valid_mask[i, :]
    if not row.any():  # all NaN → skip
        continue
    last_valid = np.where(row)[0][-1]  # index of last valid price
    # Delisting = last valid is NOT the final column (firm disappeared before end)
    if last_valid < len(dates_m) - 1:
        # Map to return column: return at dates_ret[j-1] corresponds to
        # price transition from dates_m[j-1] to dates_m[j]
        # The delisting return is for the month AFTER the last valid price
        # dates_ret has 1 fewer element than dates_m (shifted by 1)
        ret_idx = last_valid  # in dates_ret, index last_valid = return from dates_m[last_valid] to dates_m[last_valid+1]
        if ret_idx < len(dates_ret):
            returns_m.iloc[i, ret_idx] = -1.0
            delisted_firms.append({
                'ISIN': isin,
                'Name': firm_names.get(isin, 'N/A'),
                'Last_Price': dates_m[last_valid].strftime('%Y-%m'),
                'Delist_Month': dates_m[last_valid + 1].strftime('%Y-%m'),
            })

n_delisting = len(delisted_firms)
print(f"Delisting events detected: {n_delisting}")

if delisted_firms:
    df_del = pd.DataFrame(delisted_firms)
    print(f"\nFirst 15 delisted firms:")
    df_del.head(15)

## 7. Forward-Fill Missing Carbon and Revenue Data

Per the project instructions:
- **Missing between two available years** or **at the end of the sample** → use the previous year's value (forward fill).
- **Missing at the beginning** → the firm is not investable until data becomes available.

For Group AT (Scope 1+2), we compute total CO₂ = Scope 1 + Scope 2. If only one scope is available for a given firm-year, we use that value; if both are missing, the total remains NaN.

In [ ]:
def missing_report(df, name):
    n_miss = df.isna().sum().sum()
    return f"{name}: {n_miss}/{df.size} missing ({100*n_miss/df.size:.1f}%)"

print("BEFORE forward fill:")
print(f"  {missing_report(co2_s1, 'CO2 Scope 1')}")
print(f"  {missing_report(co2_s2, 'CO2 Scope 2')}")
print(f"  {missing_report(rev,    'Revenue')}")

# Forward fill along columns (years, axis=1)
co2_s1 = co2_s1.ffill(axis=1)
co2_s2 = co2_s2.ffill(axis=1)
rev    = rev.ffill(axis=1)

# Convert Revenue from thousands USD to millions USD (as required by the project)
# This is needed for Carbon Intensity = CO2 (tonnes) / Revenue (millions USD)
rev = rev / 1000.0
print('Revenue converted from Thousands USD to Millions USD')

print("\nAFTER forward fill:")
print(f"  {missing_report(co2_s1, 'CO2 Scope 1')}")
print(f"  {missing_report(co2_s2, 'CO2 Scope 2')}")
print(f"  {missing_report(rev,    'Revenue')}")

# Total CO2 = Scope 1 + Scope 2
# If either scope is NaN, total is NaN (prudent: avoids underestimating emissions)
co2_total = co2_s1 + co2_s2

print(f"\nCO2 Total (Scope 1+2): {co2_total.shape}")
print(f"  {missing_report(co2_total, 'CO2 Total')}")

## 8. Investment Set Construction

At the end of each year $Y$, we define the investment set as firms satisfying **all** of:
1. **Price available** at end of year $Y$ (RI not NaN → can invest)
2. **Sufficient return history**: ≥ 36 months of valid returns in the 10-year estimation window
3. **No stale prices**: proportion of zero-return months ≤ 50%
4. **Carbon data available**: CO₂ Scope 1+2 not NaN at end of year $Y$

We use vectorized operations to check all criteria at once.

In [ ]:
# Pre-compute December indices for quick lookup
dec_ret_idx = {d.year: i for i, d in enumerate(dates_ret) if d.month == 12}
dec_price_idx = {d.year: i for i, d in enumerate(dates_m) if d.month == 12}


def get_investment_set(year):
    """
    Vectorized investment set construction at end of year Y.
    Returns (eligible_isins, estimation_window_cols, exclusion_reasons).
    """
    # 10-year estimation window ending at Dec of year Y
    end_idx = dec_ret_idx[year]
    start_idx = max(0, end_idx - ESTIMATION_WINDOW + 1)
    window_cols = dates_ret[start_idx : end_idx + 1]
    ret_window = returns_m[window_cols]

    # Criterion 1: price available at end of year Y
    dec_col = dates_m[dec_price_idx[year]]
    has_price = ri_m[dec_col].notna()

    # Criterion 2: enough valid returns in the window
    n_valid = ret_window.notna().sum(axis=1)
    enough_returns = n_valid >= MIN_RETURN_MONTHS

    # Criterion 3: stale price filter
    n_zero = (ret_window == 0).sum(axis=1)
    # Zero proportion computed only on non-NaN returns
    zero_pct = n_zero / n_valid.replace(0, np.nan)
    not_stale = (zero_pct <= STALE_THRESHOLD) | n_valid.eq(0)

    # Criterion 4: CO2 data available
    has_carbon = co2_total[year].notna() if year in co2_total.columns else pd.Series(False, index=ri_m.index)

    # Combine all criteria
    eligible_mask = has_price & enough_returns & not_stale & has_carbon
    eligible = eligible_mask[eligible_mask].index.tolist()

    reasons = {
        'no_price':    (~has_price).sum(),
        'few_returns': (has_price & ~enough_returns).sum(),
        'stale':       (has_price & enough_returns & ~not_stale).sum(),
        'no_carbon':   (has_price & enough_returns & not_stale & ~has_carbon).sum(),
    }
    return eligible, window_cols, reasons


# Run for each year
investment_sets = {}
rows = []
for Y in range(Y0, Y_END + 1):
    elig, wcols, reasons = get_investment_set(Y)
    investment_sets[Y] = elig
    rows.append({'Year': Y, 'Eligible': len(elig), **reasons})

df_inv = pd.DataFrame(rows).set_index('Year')
df_inv.columns = ['Eligible', 'No Price', 'Few Returns', 'Stale', 'No Carbon']
df_inv

## 9. Prepare Risk-Free Rate

In [ ]:
rf_df = rf_raw.copy()
rf_df.columns = ['date_code', 'RF']
rf_df['year']  = rf_df['date_code'] // 100
rf_df['month'] = rf_df['date_code'] % 100
# RF is already a monthly percentage → convert to monthly decimal
rf_df['RF_monthly'] = rf_df['RF'] / 100

rf_dict = {(int(r['year']), int(r['month'])): r['RF_monthly']
           for _, r in rf_df.iterrows()}

print(f"Risk-free rate available for {len(rf_dict)} months")
print(f"Sample: Jan 2014 RF (monthly) = {rf_dict.get((2014, 1), 'N/A'):.6f}")
print(f"Sample: Jan 2020 RF (monthly) = {rf_dict.get((2020, 1), 'N/A'):.6f}")

## 10. Save Cleaned Data

All cleaned DataFrames are saved as CSV files so that subsequent notebooks can load them directly.

In [ ]:
OUTPUT_DIR = './cleaned_data/'
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Main datasets
ri_m.to_csv(f'{OUTPUT_DIR}ri_monthly.csv')
mv_m.to_csv(f'{OUTPUT_DIR}mv_monthly.csv')
mv_y.to_csv(f'{OUTPUT_DIR}mv_yearly.csv')
returns_m.to_csv(f'{OUTPUT_DIR}returns_monthly.csv')
co2_s1.to_csv(f'{OUTPUT_DIR}co2_scope1.csv')
co2_s2.to_csv(f'{OUTPUT_DIR}co2_scope2.csv')
co2_total.to_csv(f'{OUTPUT_DIR}co2_total.csv')
rev.to_csv(f'{OUTPUT_DIR}revenue.csv')

# Risk-free rate
rf_df[['year', 'month', 'RF', 'RF_monthly']].to_csv(f'{OUTPUT_DIR}risk_free_rate.csv', index=False)

# Investment sets (one row per year-ISIN)
inv_rows = []
for Y, isins in investment_sets.items():
    for isin in isins:
        inv_rows.append({'Year': Y, 'ISIN': isin})
pd.DataFrame(inv_rows).to_csv(f'{OUTPUT_DIR}investment_sets.csv', index=False)

# Firm names mapping
firm_names.to_csv(f'{OUTPUT_DIR}firm_names.csv')

# Helper indices
pd.Series(dec_ret_idx, name='ret_index').to_csv(f'{OUTPUT_DIR}dec_ret_indices.csv')
pd.Series(dec_price_idx, name='price_index').to_csv(f'{OUTPUT_DIR}dec_price_indices.csv')

# List of common ISINs
pd.Series(common_isins, name='ISIN').to_csv(f'{OUTPUT_DIR}common_isins.csv', index=False)

print(f"All cleaned data saved to {OUTPUT_DIR}")
print(f"Files:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_kb = os.path.getsize(f'{OUTPUT_DIR}{f}') / 1024
    print(f"  {f:.<40} {size_kb:>8.1f} KB")

## Summary

In [ ]:
print(f"""
DATA CLEANING — SUMMARY
{'='*50}
Region:                   {REGION} (North America)
Scope:                    {SCOPE}
Total firms in Static:    {static.shape[0]}
Firms in {REGION}:            {len(region_isins)}
Firms in all files:       {len(common_isins)}
Low price threshold:      {LOW_PRICE_THRESHOLD}
Stale price threshold:    {STALE_THRESHOLD*100:.0f}%
Min return months:        {MIN_RETURN_MONTHS}
Delisting events:         {n_delisting}
{'='*50}
""")

print("Investment set sizes:")
for Y in range(Y0, Y_END + 1):
    print(f"  {Y}: {len(investment_sets[Y]):>4} firms")

print(f"\nCleaned data saved to: {OUTPUT_DIR}")
print("Next step → Section 2: Portfolio Optimization")